In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv')

if 'Irrigation_Need' in orig.columns:
    orig = orig.drop(columns=['id'], errors='ignore')
    train = pd.concat([train, orig], ignore_index=True)
    print(f"Original dataset added. Total train: {len(train):,}")

print(f"Train: {train.shape} | Test: {test.shape}")

In [ ]:
def feature_engineering(df):
    df = df.copy()
    df['Is_Active_Growth']        = df['Crop_Growth_Stage'].isin(['Flowering', 'Vegetative']).astype(int)
    df['Mulching_x_ActiveGrowth'] = df['Is_Active_Growth'] * (df['Mulching_Used'] == 'No').astype(int)
    df['Evap_Stress']             = df['Temperature_C'] - df['Humidity'] - df['Rainfall_mm'] / 10
    prev_med  = df['Previous_Irrigation_mm'].median()
    moist_med = df['Soil_Moisture'].median()
    df['Water_Deficit']   = np.where((df['Previous_Irrigation_mm'] < prev_med) & (df['Soil_Moisture'] < moist_med), 1, 0)
    df['Heat_Load']       = df['Temperature_C'] * df['Sunlight_Hours']
    df['Soil_Fertility']  = df['Soil_Moisture'] * df['Organic_Carbon']
    df['Wind_Evap_Index'] = df['Wind_Speed_kmh'] * (100 - df['Humidity']) / 100
    df['Rainfall_Cat']    = pd.cut(df['Rainfall_mm'], bins=[-np.inf, 20, 50, 100, np.inf], labels=[0, 1, 2, 3]).astype(float)
    df['is_dry_soil']     = (df['Soil_Moisture'] < 25).astype(np.int8)
    df['high_risk_flag']  = (df['Is_Active_Growth'] & df['is_dry_soil']).astype(np.int8)
    df['no_mulching']     = (df['Mulching_Used'] == 'No').astype(np.int8)
    df['rain_deficit']    = (2500 - df['Rainfall_mm']).clip(lower=0)
    df['moisture_stress'] = df['rain_deficit'] / (df['Soil_Moisture'] + 1)
    df['evap_proxy']      = df['Temperature_C'] * df['Sunlight_Hours'] / (df['Humidity'] + 1)
    df['season_growth']   = df['Season'].astype(str) + '_' + df['Crop_Growth_Stage'].astype(str)
    df['source_type']     = df['Water_Source'].astype(str) + '_' + df['Irrigation_Type'].astype(str)
    df['region_soil']     = df['Region'].astype(str) + '_' + df['Soil_Type'].astype(str)
    return df

In [ ]:
X_all  = feature_engineering(train.drop(columns=['id', 'Irrigation_Need']))
X_test = feature_engineering(test.drop(columns=['id']))
y_raw  = train['Irrigation_Need']

print(f"Features: {X_all.shape[1]}")

cat_cols = X_all.select_dtypes(include='object').columns.tolist()
cat_col_indices = [X_all.columns.get_loc(c) for c in cat_cols]

X_all_cb  = X_all.copy()
X_test_cb = X_test.copy()

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_all[cat_cols]  = oe.fit_transform(X_all[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])
X_all  = X_all.astype(float)
X_test = X_test.astype(float)

label_enc = LabelEncoder()
y = label_enc.fit_transform(y_raw)
print(f"Classes: {list(label_enc.classes_)}")

classes   = np.unique(y)
N_CLASSES = len(classes)
high_idx  = label_enc.transform(['High'])[0]

class_weights     = compute_class_weight('balanced', classes=classes, y=y)
class_weight_dict = dict(zip(classes.tolist(), class_weights.tolist()))
class_weight_dict[high_idx] *= 2.0
sample_weights_all = np.array([class_weight_dict[cls] for cls in y])
print(f"Class weights: {class_weight_dict}")

In [ ]:
N_FOLDS = 5
SEED    = 42
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros((len(y), N_CLASSES))
oof_xgb  = np.zeros((len(y), N_CLASSES))
oof_cb   = np.zeros((len(y), N_CLASSES))
test_lgb = np.zeros((len(X_test), N_CLASSES))
test_xgb = np.zeros((len(X_test), N_CLASSES))
test_cb  = np.zeros((len(X_test), N_CLASSES))

In [ ]:
lgb_params = {
    'objective'        : 'multiclass',
    'num_class'        : N_CLASSES,
    'metric'           : 'multi_logloss',
    'device'           : 'gpu',
    'learning_rate'    : 0.053061334426183264,
    'num_leaves'       : 134,
    'max_depth'        : 4,
    'min_child_samples': 129,
    'feature_fraction' : 0.5821229283161274,
    'bagging_fraction' : 0.9688310483927282,
    'bagging_freq'     : 8,
    'reg_alpha'        : 0.0018227889071454036,
    'reg_lambda'       : 0.0034602344645780166,
    'n_jobs'           : -1,
    'verbose'          : -1,
    'seed'             : SEED,
}

xgb_params = {
    'objective'       : 'multi:softprob',
    'num_class'       : N_CLASSES,
    'eval_metric'     : 'mlogloss',
    'tree_method'     : 'hist',
    'device'          : 'cuda',
    'learning_rate'   : 0.05,
    'max_depth'       : 7,
    'min_child_weight': 5,
    'subsample'       : 0.8,
    'colsample_bytree': 0.8,
    'gamma'           : 0.1,
    'reg_alpha'       : 0.1,
    'reg_lambda'      : 1.0,
    'seed'            : SEED,
    'verbosity'       : 0,
}

cb_params = dict(
    loss_function='MultiClass',
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    bagging_temperature=0.5,
    random_seed=SEED,
    task_type='GPU',
    thread_count=-1,
    verbose=0,
    early_stopping_rounds=150,
    cat_features=cat_col_indices,
    class_weights={0: class_weight_dict[0], 1: class_weight_dict[1], 2: class_weight_dict[2]},
)

In [ ]:
lgb_scores, xgb_scores, cb_scores = [], [], []

print(f"\n{'='*60}")
print("LightGBM + XGBoost + CatBoost — 5-Fold (Balanced Accuracy)")
print(f"{'='*60}")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_all, y), 1):
    print(f"\n--- Fold {fold} ---")
    X_tr, X_val = X_all.iloc[trn_idx], X_all.iloc[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]
    w_tr = sample_weights_all[trn_idx]

    # LightGBM
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, free_raw_data=False)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=False)
    model_lgb = lgb.train(lgb_params, dtrain, num_boost_round=3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(500)])
    val_proba_lgb = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
    oof_lgb[val_idx] = val_proba_lgb
    test_lgb += model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration) / N_FOLDS
    lgb_score = balanced_accuracy_score(y_val, np.argmax(val_proba_lgb, axis=1))
    lgb_scores.append(lgb_score)
    print(f"LGB iter: {model_lgb.best_iteration} | Balanced Acc: {lgb_score:.5f}")

    # XGBoost
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dval_xgb   = xgb.DMatrix(X_val, label=y_val)
    model_xgb  = xgb.train(xgb_params, dtrain_xgb, num_boost_round=3000,
                            evals=[(dval_xgb, 'val')], early_stopping_rounds=150, verbose_eval=500)
    val_proba_xgb = model_xgb.predict(dval_xgb).reshape(-1, N_CLASSES)
    oof_xgb[val_idx] = val_proba_xgb
    test_xgb += model_xgb.predict(xgb.DMatrix(X_test)).reshape(-1, N_CLASSES) / N_FOLDS
    xgb_score = balanced_accuracy_score(y_val, np.argmax(val_proba_xgb, axis=1))
    xgb_scores.append(xgb_score)
    print(f"XGB iter: {model_xgb.best_iteration} | Balanced Acc: {xgb_score:.5f}")

    # CatBoost
    X_tr_cb  = X_all_cb.iloc[trn_idx]
    X_val_cb = X_all_cb.iloc[val_idx]
    train_pool = Pool(X_tr_cb, y_tr, cat_features=cat_col_indices, weight=w_tr)
    val_pool   = Pool(X_val_cb, y_val, cat_features=cat_col_indices)
    test_pool  = Pool(X_test_cb, cat_features=cat_col_indices)
    model_cb   = CatBoostClassifier(**cb_params)
    model_cb.fit(train_pool, eval_set=val_pool)
    val_proba_cb = model_cb.predict_proba(val_pool)
    oof_cb[val_idx] = val_proba_cb
    test_cb += model_cb.predict_proba(test_pool) / N_FOLDS
    cb_score = balanced_accuracy_score(y_val, np.argmax(val_proba_cb, axis=1))
    cb_scores.append(cb_score)
    print(f"CB  iter: {model_cb.best_iteration_} | Balanced Acc: {cb_score:.5f}")

print(f"\nLGB OOF: {np.mean(lgb_scores):.5f}")
print(f"XGB OOF: {np.mean(xgb_scores):.5f}")
print(f"CB  OOF: {np.mean(cb_scores):.5f}")

In [ ]:
print("Finding best ensemble weights...")
best_score, best_wl, best_wx = 0, 0.5, 0.3
for wl in np.arange(0.3, 0.8, 0.05):
    for wx in np.arange(0.1, 0.5, 0.05):
        wc = 1 - wl - wx
        if wc <= 0: continue
        blend = wl * oof_lgb + wx * oof_xgb + wc * oof_cb
        s = balanced_accuracy_score(y, np.argmax(blend, axis=1))
        if s > best_score:
            best_score, best_wl, best_wx = s, wl, wx

best_wc = 1 - best_wl - best_wx
print(f"Best weights — LGB: {best_wl:.2f} | XGB: {best_wx:.2f} | CB: {best_wc:.2f} | OOF: {best_score:.5f}")

oof_ensemble  = best_wl * oof_lgb  + best_wx * oof_xgb  + best_wc * oof_cb
test_ensemble = best_wl * test_lgb + best_wx * test_xgb + best_wc * test_cb

In [ ]:
non_high = [i for i in range(N_CLASSES) if i != high_idx]
best_thresh_score, best_thresh = 0, None
for thresh in np.arange(0.05, 0.50, 0.01):
    preds = np.array(non_high)[np.argmax(oof_ensemble[:, non_high], axis=1)]
    mask  = oof_ensemble[:, high_idx] > thresh
    preds[mask] = high_idx
    s = balanced_accuracy_score(y, preds)
    if s > best_thresh_score:
        best_thresh_score, best_thresh = s, thresh
print(f"Best threshold: {best_thresh:.2f} | OOF Balanced Acc: {best_thresh_score:.5f}")

test_pred = np.array(non_high)[np.argmax(test_ensemble[:, non_high], axis=1)]
test_pred[test_ensemble[:, high_idx] > best_thresh] = high_idx

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': label_enc.inverse_transform(test_pred)
})
submission.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv saved! Rows: {len(submission):,}")
print(submission['Irrigation_Need'].value_counts())

oof_final = np.array(non_high)[np.argmax(oof_ensemble[:, non_high], axis=1)]
oof_final[oof_ensemble[:, high_idx] > best_thresh] = high_idx
print("\nFinal OOF Report:")
print(classification_report(y, oof_final, target_names=label_enc.classes_, digits=4))